# GPU 高通量虛擬篩選 (High-Throughput Virtual Screening)
專為「藥物庫擴展」與「新穎骨架發現」設計。利用 PubChem 巨量資料庫與 GPU 叢集的強大平行運算能力，將常規的篩選時程由數天大幅壓縮至數小時內。


## Vina-GPU 的突破性改良
Vina-GPU並非由美國斯克里普斯研究所 (The Scripps Research Institute, TSRI) 的分子圖學實驗室 (Molecular Graphics Laboratory)開發，而是由南京郵電大學 (NJUPT) 的 Delta Group 聯合其他運算實驗室進行拓展。

它將最耗時的「評分函數計算」與「梯度幾何優化」演算法進行了底層重寫，封裝成 OpenCL 與 CUDA 的核心指令 (Kernels)。這項架構替換使得：

1. 千倍並行計算：原本由 CPU 依序處理的幾千個初始隨機配體姿勢 (Random Poses)，現在可以直接分配給 GPU 上的數千個流處理器 (CUDA Cores) 「同時」 進行局部最佳化與能量算分。

2. 網格數據常駐：將受體的 Grid Map 數據快取於 GPU 的高頻寬記憶體 (VRAM) 中，徹底打破了傳統 CPU 架構下的記憶體存取瓶頸 (Memory Bandwidth Bottleneck)，實現真正的高通量。

由於部分演算法的替換，雖然其能量估算仍採用Vina的物理函式，但其結果仍與CPU運算的Vina具有差異。

### Vina-GPU 引擎編譯與環境自適應
功能說明：偵測雲端 GPU 硬體拓樸（如 T4, V100, A100），動態修改 Makefile 進行 Vina-GPU 2.1 引擎的 C++ 與 OpenCL 底層編譯。

核心設計：

1. 雲端熱備份機制 (Hot-Backup)：首次成功編譯後自動鏡像至 Google Drive。後續執行若偵測到同構型硬體，將直接觸發「秒速喚醒」，免除重複編譯的效能損耗。

2. 硬體架構防呆鎖 (Architecture Interlock)：若系統偵測為純 CPU 執行階段，將觸發 RuntimeError 強制攔截，並提供標準的環境切換指引，避免高通量管線崩潰。

不過由於一些相依的函式庫暫時無法靜態編譯與備份，每次開啟新的工作階段必須要重新進行下載安裝與編譯。上述步驟都已自動化降低使用門檻。


**Hints：**
執行本工作流之前，請先登入您的 Google 帳號。後續執行時將會自動新增「MD_and_CADD」 資料夾並且將編譯執行檔備份置入其中。

In [ ]:
import os
import subprocess
import urllib.request
import tarfile
import re
import json
import shutil
import sys

# ==========================================
# 參數設定區
# ==========================================
GDRIVE_BACKUP_DIR = "/content/drive/MyDrive/MD_and_CADD/VinaGPU_Backups"
OPENCL_BACKUP_DIR = f"{GDRIVE_BACKUP_DIR}/OpenCL"
META_FILE = f"{GDRIVE_BACKUP_DIR}/hardware_meta.json"
FINAL_BIN_NAME = "AutoDock-Vina-GPU-2-1"

# ==========================================
# 輔助函式區
# ==========================================
def get_gpu_info():
    """檢測 GPU 狀態，回傳架構代號(如 '75')與顯卡名稱(如 'Tesla T4')"""
    try:
        cc_raw = subprocess.check_output("nvidia-smi --query-gpu=compute_cap --format=csv,noheader", shell=True, text=True).strip()
        name_raw = subprocess.check_output("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, text=True).strip()
        return cc_raw.replace('.', ''), name_raw
    except:
        return None, None

def mount_gdrive():
    """靜默掛載 Google Drive"""
    if not os.path.exists("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount('/content/drive')
    os.makedirs(GDRIVE_BACKUP_DIR, exist_ok=True)

def safe_extract(tar, path="."):
    """完美解決 Python 3.14 解壓縮警告的相容性函式"""
    if hasattr(tarfile, 'data_filter'):
        tar.extractall(path, filter='data')
    else:
        tar.extractall(path)

def install_minimal_deps():
    """可攜式喚醒所需的最少相依性 (約 15 秒)"""
    print("[系統] 正在安裝極簡執行環境 (Boost & CUDA 動態庫)...")
    subprocess.run("apt-get update -q && apt-get install -y nvidia-cuda-toolkit libboost-all-dev", shell=True, stdout=subprocess.DEVNULL)

def compile_vina_gpu(gpu_cc):
    """執行完整編譯流程"""
    print("[編譯] 開始原始碼編譯流程 (約需 2~3 分鐘)...")
    install_minimal_deps()

    # 1. 準備 Boost 原始碼 (僅編譯時需要)
    os.chdir("/content")
    if not os.path.exists("boost_1_84_0/boost"):
        urllib.request.urlretrieve("https://archives.boost.io/release/1.84.0/source/boost_1_84_0.tar.gz", "boost.tar.gz")
        with tarfile.open("boost.tar.gz", "r:gz") as tar:
            safe_extract(tar)
        os.remove("boost.tar.gz")

    # 2. 獲取 Vina-GPU 原始碼
    subprocess.run("rm -rf Vina-GPU-2.1_src && git clone -q https://github.com/DeltaGroupNJUPT/Vina-GPU-2.1.git Vina-GPU-2.1_src", shell=True)
    work_dir = "/content/Vina-GPU-2.1_src/AutoDock-Vina-GPU-2.1"
    os.chdir(work_dir)

    # 3. 絕對路徑修復手術 (避免編譯器迷路)
    with open("Makefile", "r") as f: mk = f.read()
    mk = re.sub(r'arch=compute_\d+,code=sm_\d+', f'arch=compute_{gpu_cc},code=sm_{gpu_cc}', mk)
    mk = mk.replace("../../boost_1_84_0", "/content/boost_1_84_0").replace("../boost_1_84_0", "/content/boost_1_84_0")
    with open("Makefile", "w") as f: f.write(mk)

    # 4. 強制補完 Include 路徑並編譯
    env_vars = f"export CPLUS_INCLUDE_PATH={work_dir}/lib:{work_dir}/main:{work_dir}/OpenCL/inc:/content/boost_1_84_0 && "
    subprocess.run(env_vars + "make clean > /dev/null && make source > /dev/null", shell=True)

    if os.path.exists(FINAL_BIN_NAME):
        subprocess.run(f"mv {FINAL_BIN_NAME} /content/ && chmod +x /content/{FINAL_BIN_NAME}", shell=True)
        # 將編譯出來的 OpenCL 藍圖放一份在 /content，供執行與備份使用
        if not os.path.exists("/content/OpenCL"):
            shutil.copytree(f"{work_dir}/OpenCL", "/content/OpenCL")
        return True
    return False

# ==========================================
# 主邏輯區
# ==========================================
print("[啟動] 正在分析 Colab 硬體配置...")
gpu_cc, gpu_name = get_gpu_info()

if not gpu_cc:
    # ★ 修改處：直接擋下 CPU 環境，並給予明確的切換指示
    print("[判定] 偵測為純 CPU 執行階段。")
    print("[錯誤] 本工作流專為 Vina-GPU 加速設計，後續指令無法在 CPU 環境下執行。")
    print("[提示] 請點選上方選單：「執行階段 (Runtime)」 -> 「變更執行階段類型 (Change runtime type)」，")
    print("[提示] 將硬體加速器切換為「T4 GPU」或更高等級，然後重新執行本儲存格。")
    raise RuntimeError("缺少 GPU 硬體加速器，執行強制終止。")

else:
    print(f"[判定] 偵測到 GPU 加速器: {gpu_name} (架構: sm_{gpu_cc})")
    mount_gdrive()

    backup_bin_path = f"{GDRIVE_BACKUP_DIR}/{FINAL_BIN_NAME}_sm{gpu_cc}"

    if os.path.exists(backup_bin_path):
        print(f"[紀錄] 於雲端硬碟尋獲相符之架構編譯檔 (sm_{gpu_cc})。")
        install_minimal_deps()
        shutil.copy(backup_bin_path, f"/content/{FINAL_BIN_NAME}")
        subprocess.run(f"chmod +x /content/{FINAL_BIN_NAME}", shell=True)

        # 將 OpenCL 藍圖從雲端倒回 /content
        if os.path.exists(OPENCL_BACKUP_DIR):
            if os.path.exists("/content/OpenCL"):
                shutil.rmtree("/content/OpenCL")
            shutil.copytree(OPENCL_BACKUP_DIR, "/content/OpenCL")
        else:
            subprocess.run("git clone -q https://github.com/DeltaGroupNJUPT/Vina-GPU-2.1.git /tmp/vina_src", shell=True)
            shutil.copytree("/tmp/vina_src/AutoDock-Vina-GPU-2.1/OpenCL", "/content/OpenCL", dirs_exist_ok=True)
            shutil.copytree("/content/OpenCL", OPENCL_BACKUP_DIR, dirs_exist_ok=True)

        print("[完成] GPU 引擎與 OpenCL 藍圖已自備份庫喚醒，準備就緒。")

    else:
        print(f"[提示] 未尋獲 sm_{gpu_cc} 之備份檔，觸發重新編譯機制...")
        success = compile_vina_gpu(gpu_cc)

        if success:
            shutil.copy(f"/content/{FINAL_BIN_NAME}", backup_bin_path)

            # 第一次編譯成功後，將 OpenCL 藍圖存上雲端
            if not os.path.exists(OPENCL_BACKUP_DIR):
                shutil.copytree("/content/OpenCL", OPENCL_BACKUP_DIR)

            meta_data = {}
            if os.path.exists(META_FILE):
                with open(META_FILE, 'r') as f: meta_data = json.load(f)

            meta_data[f"sm_{gpu_cc}"] = {
                "gpu_name": gpu_name,
                "binary_file": f"{FINAL_BIN_NAME}_sm{gpu_cc}",
                "status": "compiled_successfully"
            }
            with open(META_FILE, 'w') as f: json.dump(meta_data, f, indent=4)

            print(f"[完成] 編譯成功！檔案與 OpenCL 藍圖已備份至: {GDRIVE_BACKUP_DIR}")
            print("[紀錄] 硬體配置已更新至 JSON 紀錄檔。")
        else:
            print("[錯誤] 編譯失敗，請檢查原始碼或相依性。")

os.chdir("/content")

[啟動] 正在分析 Colab 硬體配置...
[判定] 偵測到 GPU 加速器: Tesla T4 (架構: sm_75)
Mounted at /content/drive
[紀錄] 於雲端硬碟尋獲相符之架構編譯檔 (sm_75)。
[系統] 正在安裝極簡執行環境 (Boost & CUDA 動態庫)...
[完成] GPU 引擎與 OpenCL 藍圖已自備份庫喚醒，準備就緒。


### 專案初始化與基準分子準備
功能說明：掛載受體與「基準配體 (Reference Ligand)」，初始化標準化結果目錄 (/results)。

核心設計：實作的「Regex 字串切片對齊法 (Regex-based String Alignment)」。在轉換基準分子為 PDBQT 時，系統能強健地吸收第三方軟體產生的浮點數捨入誤差與排版錯位，確保座標矩陣與原子屬性 100% 精準繼承。

In [ ]:
# 隱藏安裝輸出來保持版面整潔
!pip install -q meeko py3Dmol ipywidgets biopython rdkit requests pandas gemmi prody
!apt-get install -q -y openbabel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.0/294.0 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.5/37.5 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 72.9 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libinchi1 libmaeparser1 libopenbabel7
The following NEW packages will be installed:
  libinchi1 libmaeparser1 libopenbabel7 openbabel
0 upgraded, 4 newly installed, 0 to remove and 56 not upgraded.
Need to get 3,903 kB of archives.
After this operation, 16.9 MB of additional disk space will be used.
Get:1 http:/

In [ ]:
import os
import time
import shutil
import subprocess
import requests
import numpy as np
import pandas as pd
from google.colab import files
from rdkit import Chem
from rdkit.Chem import AllChem

print("[系統] 模組載入成功。")

[系統] 模組載入成功。


In [ ]:
# ---------------------------------------------------------
# 全域共用函數：Ligand 完美標準化與三檔案 (.pdb, .mol2, .pdbqt) 輸出
# ---------------------------------------------------------
def patch_pdbqt_from_pdb(pdbqt_path, pdb_path):
    pdb_atoms = []
    with open(pdb_path, 'r') as f:
        for line in f:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                try:
                    x = float(line[30:38])
                    y = float(line[38:46])
                    z = float(line[46:54])
                    metadata_block = line[11:27]
                    pdb_atoms.append({'x': x, 'y': y, 'z': z, 'block': metadata_block})
                except ValueError:
                    pass

    with open(pdbqt_path, 'r') as f:
        pdbqt_lines = f.readlines()

    new_lines = []
    for line in pdbqt_lines:
        if line.startswith("ATOM") or line.startswith("HETATM"):
            try:
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])

                min_dist = 999.0
                best_block = line[11:27]

                for patom in pdb_atoms:
                    dist = (x - patom['x'])**2 + (y - patom['y'])**2 + (z - patom['z'])**2
                    if dist < min_dist:
                        min_dist = dist
                        best_block = patom['block']

                if min_dist < 0.01:
                    line = line[:11] + best_block + line[27:]
            except ValueError:
                pass
        new_lines.append(line)

    with open(pdbqt_path, 'w') as f:
        f.write("".join(new_lines))

def process_and_save_ligand(mol, output_prefix, res_name="LIG"):
    if mol is None: return False

    mol = Chem.AddHs(mol, addCoords=True)
    element_counts = {}
    for atom in mol.GetAtoms():
        symbol = atom.GetSymbol()
        element_counts[symbol] = element_counts.get(symbol, 0) + 1
        atom_name = f"{symbol}{element_counts[symbol]}"

        info = Chem.AtomPDBResidueInfo()
        info.SetName(atom_name.ljust(4))
        info.SetResidueName(res_name)
        info.SetResidueNumber(1)
        info.SetIsHeteroAtom(True)
        atom.SetMonomerInfo(info)

    pdb_out = f"{output_prefix}.pdb"
    Chem.MolToPDBFile(mol, pdb_out)

    mol2_out = f"{output_prefix}.mol2"
    subprocess.run(f'obabel -i pdb "{pdb_out}" -o mol2 -O "{mol2_out}" --title "{res_name}"', shell=True, stderr=subprocess.DEVNULL)

    pdbqt_out = f"{output_prefix}.pdbqt"
    subprocess.run(f'mk_prepare_ligand.py -i "{mol2_out}" -o "{pdbqt_out}"', shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    if os.path.exists(pdb_out) and os.path.exists(pdbqt_out):
        patch_pdbqt_from_pdb(pdbqt_out, pdb_out)
    else:
        return False

    return True

In [ ]:
# ---------------------------------------------------------
# 專案目錄與檔案上傳處理
# ---------------------------------------------------------
os.chdir("/content")
PROJ_DIR = "/content/HTVS_Project"

if os.path.exists(PROJ_DIR):
    shutil.rmtree(PROJ_DIR)
os.makedirs(f"{PROJ_DIR}/results", exist_ok=True)
os.makedirs(f"{PROJ_DIR}/results/ligand_user_upload", exist_ok=True)

print("\n[步驟 1/2] 請上傳 目標受體 Receptor (.pdb 或 .cif)")
uploaded_rec = files.upload()
raw_rec = list(uploaded_rec.keys())[0]
rec_ext = raw_rec.split('.')[-1].lower()
rec_path = f"{PROJ_DIR}/receptor_original.{rec_ext}"
os.rename(raw_rec, rec_path)

print("\n[步驟 2/2] 請上傳 參考配體 Reference Ligand (.sdf, .mol2 或 .pdb)")
uploaded_lig = files.upload()
raw_lig = list(uploaded_lig.keys())[0]
lig_ext = raw_lig.split('.')[-1].lower()
lig_path = f"{PROJ_DIR}/ligand_original.{lig_ext}"
os.rename(raw_lig, lig_path)


[步驟 1/2] 請上傳 目標受體 Receptor (.pdb 或 .cif)


Saving receptor.pdb to receptor.pdb

[步驟 2/2] 請上傳 參考配體 Reference Ligand (.sdf, .mol2 或 .pdb)


Saving Conformer3D_COMPOUND_CID_5287836 (1).sdf to Conformer3D_COMPOUND_CID_5287836 (1).sdf


In [ ]:
# --- 轉換 Receptor ---
actual_rec = rec_path
if rec_ext == 'cif':
    subprocess.run(f'obabel -i cif "{rec_path}" -o pdb -O "{PROJ_DIR}/temp_rec.pdb"', shell=True)
    actual_rec = f"{PROJ_DIR}/temp_rec.pdb"

cmd_rec = f'mk_prepare_receptor.py -i "{actual_rec}" --write_pdbqt "{PROJ_DIR}/receptor.pdbqt"'
res_rec = subprocess.run(cmd_rec, shell=True, capture_output=True, text=True)

if os.path.exists(f"{PROJ_DIR}/receptor.pdbqt"):
    print("\n[系統] 受體轉換完成: receptor.pdbqt")
else:
    raise FileNotFoundError("[錯誤] 找不到 receptor.pdbqt，請檢查日誌。")

# --- 轉換 Reference Ligand ---
subprocess.run(f'obabel "{lig_path}" -o sdf -O "{PROJ_DIR}/temp_ref.sdf"', shell=True)
ref_supplier = Chem.SDMolSupplier(f'{PROJ_DIR}/temp_ref.sdf', removeHs=False)
ref_mol = ref_supplier[0] if len(ref_supplier) > 0 else None

if ref_mol is None:
    raise ValueError("[錯誤] RDKit 解析配體失敗，請檢查上傳的檔案結構。")

mol_no_h = Chem.RemoveHs(ref_mol)
ref_smiles = Chem.MolToSmiles(mol_no_h, isomericSmiles=True)

out_prefix = f"{PROJ_DIR}/results/ligand_user_upload/ligand_user_upload"
if process_and_save_ligand(ref_mol, out_prefix):
    print(f"[系統] 參考配體準備完畢 (已生成標準化 .pdb, .mol2, .pdbqt)")
    print(f"[系統] SMILES: {ref_smiles}")
else:
    raise FileNotFoundError("[錯誤] 參考配體轉換失敗。")


[系統] 受體轉換完成: receptor.pdbqt
[系統] 參考配體準備完畢 (已生成標準化 .pdb, .mol2, .pdbqt)
[系統] SMILES: [H]/N=C(\N)c1cccc(C(=O)N[C@@H](C(=O)N2CCC(C(=O)c3ccccc3)CC2)c2ccccc2)c1


### 基準分子對接與「真實質心」提取 (Sandbox 模式)
功能說明：利用 GPU 進行基準分子的首次極速對接測試。

核心設計：導入「質心動態校正 (Dynamic Centroid Calibration)」機制。系統會自動解析最佳預測姿態，重新計算其「真實幾何中心 (Centroid)」。後續龐大的候選分子陣列將統一錨定此校正後的質心，極大化降低配體脫離結合口袋的假陰性率。

In [ ]:
VINA_GPU_PATH = "./AutoDock-Vina-GPU-2-1"
os.chdir("/content")

print("[輸入] 請輸入基準對接的 Grid Box 參數：")
try:
    cx = float(input("中心 X (cx): "))
    cy = float(input("中心 Y (cy): "))
    cz = float(input("中心 Z (cz): "))
    sx = float(input("長度 X (size_x) [預設 30]: ") or 30.0)
    sy = float(input("寬度 Y (size_y) [預設 30]: ") or 30.0)
    sz = float(input("高度 Z (size_z) [預設 30]: ") or 30.0)
except ValueError:
    print("[提示] 輸入錯誤，使用預設值 (0,0,0 / 30,30,30)")
    cx, cy, cz, sx, sy, sz = 0.0, 0.0, 0.0, 30.0, 30.0, 30.0

for bin_file in ["/content/Kernel1_Opt.bin", "/content/Kernel2_Opt.bin"]:
    if os.path.exists(bin_file):
        os.remove(bin_file)

print(f"\n[啟動] GPU 引擎進行基準對接 (尺寸: {sx} x {sy} x {sz}) ...")

[輸入] 請輸入基準對接的 Grid Box 參數：
中心 X (cx): 15
中心 Y (cy): 0.5
中心 Z (cz): 20
長度 X (size_x) [預設 30]: 30
寬度 Y (size_y) [預設 30]: 30
高度 Z (size_z) [預設 30]: 30

[啟動] GPU 引擎進行基準對接 (尺寸: 30.0 x 30.0 x 30.0) ...


In [ ]:
shutil.copy(f"{PROJ_DIR}/receptor.pdbqt", "/content/run_receptor.pdbqt")
shutil.copy(f"{PROJ_DIR}/results/ligand_user_upload/ligand_user_upload.pdbqt", "/content/run_ligand_user_upload.pdbqt")

cmd = [
    VINA_GPU_PATH,
    "--receptor", "run_receptor.pdbqt",
    "--ligand", "run_ligand_user_upload.pdbqt",
    "--center_x", str(cx), "--center_y", str(cy), "--center_z", str(cz),
    "--size_x", str(sx), "--size_y", str(sy), "--size_z", str(sz),
    "--thread", "8000"
]
subprocess.run(cmd, capture_output=True)

final_out = f"{PROJ_DIR}/results/ligand_user_upload/ligand_user_upload_out_pre.pdbqt"
if os.path.exists("/content/run_ligand_user_upload_out.pdbqt"):
    shutil.move("/content/run_ligand_user_upload_out.pdbqt", final_out)

ref_affinity = 0.0
new_cx, new_cy, new_cz = cx, cy, cz
coords = []

if os.path.exists(final_out):
    with open(final_out, "r") as f:
        in_pose_1 = False
        for line in f:
            if line.startswith("REMARK VINA RESULT") and ref_affinity == 0.0:
                ref_affinity = float(line.split()[3])
            if line.startswith("MODEL 1"): in_pose_1 = True
            if line.startswith("ENDMDL"): in_pose_1 = False
            if in_pose_1 and (line.startswith("ATOM") or line.startswith("HETATM")):
                coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])

    if coords:
        new_cx, new_cy, new_cz = np.mean(coords, axis=0)
        print(f"\n[系統] 初步對接完成。最佳 Affinity: {ref_affinity} kcal/mol")
        print(f"[系統] 提取真實質心供後續使用: X={new_cx:.2f}, Y={new_cy:.2f}, Z={new_cz:.2f}")
    else:
        print("\n[錯誤] 解析 Pose 1 原子座標失敗。")
else:
    print(f"\n[錯誤] 對接失敗，未產生結果檔案。")


[系統] 初步對接完成。最佳 Affinity: -9.0 kcal/mol
[系統] 提取真實質心供後續使用: X=12.58, Y=-5.23, Z=22.66


### PubChem 智慧檢索與雙重指紋相似度分析
功能說明：以上傳分子為基準，介接 PubChem REST API 執行結構相似度擴展與篩選。

核心設計：動態門檻下探 (Dynamic Threshold Descent)：自 85% 相似度啟動檢索，若庫存不足則自動步進式下調門檻。整合 Async 非同步輪詢機制，徹底避開 API 阻斷 (Rate Limiting) 風險。


In [ ]:
# ==============自訂區===============
current_threshold = 85
TOP_N_CANDIDATES = 50   # ★ 最後要保留進入對接的最強候選分子數量
target_min_count = 300  # ★ 這是「評估停止下探」的底線數量，不是 API 的截斷數量
SEARCH_MODE = "async"  # 可以填入 "async" 或 "fast"，"fast"提供更快速但非全pubchem分子庫搜尋，"async"則提供全面搜尋
# ==============自訂區===============

In [ ]:
import requests
import time
import pandas as pd
from IPython.display import display
from rdkit import Chem
from rdkit.Chem import DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import MACCSkeys

def pubchem_fast_search(cid, threshold=90, max_records=5000):
    print(f"  [Fast Search] 提交同步搜尋 (Threshold={threshold}%)...")
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/fastsimilarity_2d/cid/{cid}/cids/JSON?Threshold={threshold}&MaxRecords={max_records}"
    try:
        res = requests.get(url, timeout=35)
        if res.status_code == 200:
            return res.json().get('IdentifierList', {}).get('CID', [])
        else:
            print(f"  [錯誤] 搜尋失敗 (狀態碼: {res.status_code})")
    except Exception as e:
        print(f"  [警告] Fast Search 發生逾時或錯誤: {e}")
    return []

def pubchem_async_search(cid, threshold=90, max_records=5000):
    print(f"  [Async Search] 提交非同步任務 (Threshold={threshold}%)...")
    url_submit = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/similarity/cid/{cid}/JSON?Threshold={threshold}&MaxRecords={max_records}"
    try:
        res_submit = requests.get(url_submit, timeout=20)
        if res_submit.status_code == 200:
            print("  [系統] 伺服器快取命中，瞬間完成。")
            return res_submit.json().get('IdentifierList', {}).get('CID', [])
        elif res_submit.status_code == 202:
            listkey = res_submit.json().get('Waiting', {}).get('ListKey')
            print(f"  [狀態] 進入排隊序列 (ListKey: {listkey})，持續輪詢中", end="")
            while True:
                time.sleep(8)
                res_poll = requests.get(f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/listkey/{listkey}/cids/JSON", timeout=20)
                if res_poll.status_code == 200:
                    print("\n  [系統] 伺服器深度運算完成。")
                    return res_poll.json().get('IdentifierList', {}).get('CID', [])
                elif res_poll.status_code == 202:
                    print(".", end="", flush=True)
                else:
                    print(f"\n  [錯誤] 狀態查詢失敗 (狀態碼: {res_poll.status_code})")
                    break
        else:
            print(f"  [錯誤] 任務提交失敗 (狀態碼: {res_submit.status_code})")
    except Exception as e:
        print(f"\n  [警告] Async Search 發生錯誤: {e}")
    return []

print("--- 步驟 1: 解析目標分子取得 CID ---")
url_target = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/cids/JSON"
res_target = requests.post(url_target, data={'smiles': ref_smiles})

if res_target.status_code == 200 and 'IdentifierList' in res_target.json():
    target_cid = res_target.json()['IdentifierList']['CID'][0]
    print(f"[系統] 成功鎖定目標分子 CID: {target_cid}\n")
else:
    raise ValueError("[錯誤] 無法從 PubChem 獲取目標分子的 CID。")

time.sleep(0.5)

--- 步驟 1: 解析目標分子取得 CID ---
[系統] 成功鎖定目標分子 CID: 5287836



In [ ]:
print("--- 步驟 2: 智慧動態相似度搜尋 ---")


cids_2d = []
while current_threshold >= 60:
    if SEARCH_MODE == "async":
        cids_2d = pubchem_async_search(target_cid, threshold=current_threshold, max_records=1000)
    else:
        cids_2d = pubchem_fast_search(target_cid, threshold=current_threshold, max_records=1000)

    print(f"[系統] 門檻 {current_threshold}% 撈回 {len(cids_2d)} 個相似 CID。")

    if len(cids_2d) >= target_min_count:
        print(f"[系統] 數量已達分析池標準 (>= {target_min_count})，停止擴大搜尋。")
        break
    else:
        print(f"[提示] 數量小於 {target_min_count}，自動調降門檻 5% 以擴大範圍...\n")
        current_threshold -= 5
        time.sleep(2)

print("\n--- 步驟 3: 過濾純度 (排除混合物/鹽類) ---")
final_results = []
chunk_size = 50  # 這是批次發送 API 的大小，不影響您的候選總數

for i in range(0, len(cids_2d), chunk_size):
    chunk = cids_2d[i : i + chunk_size]
    cid_str = ",".join(map(str, chunk))
    url_prop = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid_str}/property/CovalentUnitCount,ConformerCount3D/JSON"

    try:
        res_prop = requests.get(url_prop).json()
        properties = res_prop.get('PropertyTable', {}).get('Properties', [])
        for prop in properties:
            if prop.get('CovalentUnitCount', 0) == 1:
                final_results.append({
                    'CID': prop['CID'],
                    'Has_3D': prop.get('ConformerCount3D', 0) > 0
                })
    except: pass
    time.sleep(0.2)

candidate_cids = [res['CID'] for res in final_results]
print(f"[系統] 從分析池中留存 {len(candidate_cids)} 個純物質候選分子。")

print("\n--- 步驟 4: 批次下載 SMILES 與 RDKit 雙重相似度精算 ---")
cid_to_smiles = {}
for i in range(0, len(candidate_cids), chunk_size):
    chunk = candidate_cids[i : i + chunk_size]
    cid_str = ",".join(map(str, chunk))
    url_smiles = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid_str}/property/IsomericSMILES,SMILES/JSON"
    try:
        res_smiles = requests.get(url_smiles).json()
        properties = res_smiles.get('PropertyTable', {}).get('Properties', [])
        for prop in properties:
            comp_smiles = prop.get('IsomericSMILES', prop.get('SMILES', ''))
            if comp_smiles: cid_to_smiles[prop['CID']] = comp_smiles
    except: pass
    time.sleep(0.2)

mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
target_mol = Chem.MolFromSmiles(ref_smiles)
target_fp_morgan = mfpgen.GetFingerprint(target_mol)
target_fp_maccs = MACCSkeys.GenMACCSKeys(target_mol)

data_for_df = []
for res in final_results:
    cid = res['CID']
    raw_smiles = cid_to_smiles.get(cid, '')
    if not raw_smiles: continue

    mol = Chem.MolFromSmiles(raw_smiles)
    if not mol: continue

    canonical_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
    comp_fp_morgan = mfpgen.GetFingerprint(mol)
    sim_morgan = DataStructs.TanimotoSimilarity(target_fp_morgan, comp_fp_morgan)
    comp_fp_maccs = MACCSkeys.GenMACCSKeys(mol)
    sim_maccs = DataStructs.TanimotoSimilarity(target_fp_maccs, comp_fp_maccs)

    data_for_df.append({
        'CID': cid,
        'smiles': canonical_smiles,
        'sim_morgan': sim_morgan,
        'sim_maccs': sim_maccs,
        'Has_3D': res['Has_3D']
    })

df_candidates = pd.DataFrame(data_for_df)
if not df_candidates.empty:
    df_candidates = df_candidates[df_candidates['sim_morgan'] < 1.0]
    df_candidates = df_candidates.sort_values(by='sim_morgan', ascending=False).reset_index(drop=True)

    # ★ 改用自訂參數，動態決定抽取數量
    top_candidates = df_candidates.head(TOP_N_CANDIDATES)

    print(f"\n[系統] 深度分析與去重完成。已從池中選出前 {len(top_candidates)} 名最佳候選分子。")
    print("[提示] 請檢查下方表格的變數，確認無誤後再執行 Cell 3-B：")

    df_display = top_candidates.copy()
    df_display['sim_morgan'] = df_display['sim_morgan'].apply(lambda x: f"{x*100:.2f}%")
    df_display['sim_maccs'] = df_display['sim_maccs'].apply(lambda x: f"{x*100:.2f}%")
    df_display = df_display.rename(columns={
        "sim_morgan": "Morgan Similarity",
        "sim_maccs": "MACCS Similarity",
        "smiles": "RDKit Standardized SMILES",
        "Has_3D": "Has PubChem 3D"
    })
    display(df_display)
else:
    print("[錯誤] 找不到任何符合條件的分子。")
    top_candidates = pd.DataFrame()

--- 步驟 2: 智慧動態相似度搜尋 ---
  [Async Search] 提交非同步任務 (Threshold=85%)...
  [狀態] 進入排隊序列 (ListKey: 1139650318379172702)，持續輪詢中
  [系統] 伺服器深度運算完成。
[系統] 門檻 85% 撈回 632 個相似 CID。
[系統] 數量已達分析池標準 (>= 300)，停止擴大搜尋。

--- 步驟 3: 過濾純度 (排除混合物/鹽類) ---
[系統] 從分析池中留存 612 個純物質候選分子。

--- 步驟 4: 批次下載 SMILES 與 RDKit 雙重相似度精算 ---

[系統] 深度分析與去重完成。已從池中選出前 50 名最佳候選分子。
[提示] 請檢查下方表格的變數，確認無誤後再執行 Cell 3-B：


,CID,RDKit Standardized SMILES,Morgan Similarity,MACCS Similarity,Has PubChem 3D
0,5287836,N=C(N)c1cccc(C(=O)N[C@@H](C(=O)N2CCC(C(=O)c3cc...,87.04%,100.00%,True
1,1631,N=C(N)c1cccc(C(=O)NC(C(=O)N2CCC(C(=O)c3ccccc3)...,87.04%,100.00%,True
2,10413130,N=C(N)c1cccc(C(=O)N[C@@H](C(=O)N2CCC(C(=O)c3cc...,78.33%,91.84%,True
3,133792590,O=C(NC(C(=O)N1CCC(C(=O)c2ccccc2)CC1)c1ccccc1)c...,73.08%,84.44%,True
4,87534536,N/N=C\c1cccc(C(=O)NC(C(=O)N2CCC(C(=O)c3ccccc3)...,66.67%,85.71%,True
5,44376754,N=C(N)c1cccc(C(=O)N[C@@H](C(=O)N2CCC(Cc3ccccc3...,66.13%,93.48%,True
6,44376712,N=C(N)c1cccc(C(=O)N[C@@H](C(=O)N2CCN(C(=O)c3cc...,64.41%,95.65%,True
7,101104399,N=C(N)c1cccc(C(=O)N[C@H](Cc2ccccc2)C(=O)N2CCC(...,63.49%,95.74%,True
8,44376651,N=C(N)c1cccc(C(=O)N[C@@H](C(=O)N2CCN(c3ccccc3)...,62.30%,87.76%,True
9,445296,N=C(N)c1cccc(C(=O)N[C@@H](C(=O)N2C=C[C@@H](C(=...,61.54%,87.23%,True


### 結構建立與強固型轉換
功能說明：批次為二維候選清單生成三維空間座標，並產出對接專用的 .pdbqt 檔案。

核心設計：建構「雙層容錯處理架構 (Dual-Layer Fallback Architecture)」。
系統優先嘗試提取 PubChem 官方驗證之 3D 結構；若遇缺失，則無縫切換至 RDKit ETKDGv3 演算法自主折疊 3D 構象，並搭配 MMFF94 力場執行幾何優化，確保高通量管線不會因單一分子結構異常而中斷。

In [ ]:
import os
import time
import requests
from rdkit import Chem
from rdkit.Chem import AllChem

print("--- 步驟 5: 3D 結構建立與原子專一化處理 ---")
valid_compounds = []

if 'top_candidates' not in locals() or top_candidates.empty:
    raise ValueError("[錯誤] 找不到候選清單，請確認 Cell 3-A 是否已成功執行並找到分子。")

for index, row in top_candidates.iterrows():
    cid = row['CID']
    has_3d = row['Has_3D']
    smiles_str = row['smiles']
    sim_morgan = row['sim_morgan']
    sim_maccs = row['sim_maccs']

    mol_id = f"ligand_CID_{cid}"
    mol_dir = f"{PROJ_DIR}/results/{mol_id}"
    os.makedirs(mol_dir, exist_ok=True)
    out_prefix = f"{mol_dir}/{mol_id}"

    print(f"  -> [{index+1}/{len(top_candidates)}] 正在處理 {mol_id}...", end=" ")
    mol_3d = None
    source = ""

    # 情況 A：優先下載 PubChem 3D
    if has_3d:
        url_sdf = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/SDF?record_type=3d"
        try:
            res_sdf = requests.get(url_sdf, timeout=10)
            if res_sdf.status_code == 200:
                mol_3d = Chem.MolFromMolBlock(res_sdf.text, removeHs=False)
                if mol_3d:
                    source = "PubChem 下載"
        except Exception:
            pass # 改用 Exception，避免吞掉 KeyboardInterrupt

    # 情況 B：PubChem 沒有 3D 或下載失敗，使用 RDKit 生成
    if mol_3d is None:
        try:
            mol_2d = Chem.MolFromSmiles(smiles_str)
            if mol_2d:
                mol_with_h = Chem.AddHs(mol_2d)
                # ★ 修正：檢查 3D 生成是否成功 (0 代表成功)
                embed_status = AllChem.EmbedMolecule(mol_with_h, AllChem.ETKDGv3())
                if embed_status == 0:
                    AllChem.MMFFOptimizeMolecule(mol_with_h)
                    mol_3d = mol_with_h
                    source = "RDKit 生成"
        except Exception:
            pass

    # 情況 C：呼叫全域函數進行存檔與 PDBQT 轉換
    if mol_3d:
        success = process_and_save_ligand(mol_3d, out_prefix)
        if success:
            valid_compounds.append({
                "id": mol_id,
                "cid": str(cid),
                "smiles": smiles_str,
                "sim_morgan": sim_morgan,
                "sim_maccs": sim_maccs,
                "pdbqt_path": f"{out_prefix}.pdbqt"
            })
            print(f"[{source}完成]")
        else:
            print(f"[錯誤] 檔案轉換失敗")
    else:
        print(f"[錯誤] 3D 結構生成失敗")

    time.sleep(0.2)

print(f"\n[系統] 成功準備了 {len(valid_compounds)} 個分子的標準化檔案。")
print("[提示] 請執行 Cell 4 進行批次對接。")

--- 步驟 5: 3D 結構建立與原子專一化處理 ---
  -> [1/50] 正在處理 ligand_CID_5287836... [PubChem 下載完成]
  -> [2/50] 正在處理 ligand_CID_1631... [PubChem 下載完成]
  -> [3/50] 正在處理 ligand_CID_10413130... [PubChem 下載完成]
  -> [4/50] 正在處理 ligand_CID_133792590... [PubChem 下載完成]
  -> [5/50] 正在處理 ligand_CID_87534536... [PubChem 下載完成]
  -> [6/50] 正在處理 ligand_CID_44376754... [PubChem 下載完成]
  -> [7/50] 正在處理 ligand_CID_44376712... [PubChem 下載完成]
  -> [8/50] 正在處理 ligand_CID_101104399... [PubChem 下載完成]
  -> [9/50] 正在處理 ligand_CID_44376651... [PubChem 下載完成]
  -> [10/50] 正在處理 ligand_CID_445296... [PubChem 下載完成]
  -> [11/50] 正在處理 ligand_CID_44376713... [PubChem 下載完成]
  -> [12/50] 正在處理 ligand_CID_101104400... [PubChem 下載完成]
  -> [13/50] 正在處理 ligand_CID_55964906... [PubChem 下載完成]
  -> [14/50] 正在處理 ligand_CID_10253674... [PubChem 下載完成]
  -> [15/50] 正在處理 ligand_CID_56439778... [PubChem 下載完成]
  -> [16/50] 正在處理 ligand_CID_25519802... [PubChem 下載完成]
  -> [17/50] 正在處理 ligand_CID_24238308... [PubChem 下載完成]
  -> [18/50] 正在處理 ligand_CID_25

### GPU 批次高通量對接核心
功能說明：將處理完畢的候選分子陣列送入 Vina-GPU 引擎，啟動數千條執行緒 (Threads) 的超大規模平行運算。

核心設計：實裝「沙盒隔離與例外攔截 (Sandbox Isolation & Exception Handling)」。
每個分子皆於獨立的 I/O 暫存區執行。若特定配體因空間位阻極端異常導致 Vina 引擎無法產出結果，攔截器將優雅地捕獲異常、賦予防呆極值 (Affinity = 999.0)，並確保其餘數百個分子的對接進程順暢無阻。

In [ ]:
os.chdir("/content")
import os
import shutil
import subprocess

ligand_user_data = {
    "id": "ligand_user_upload",
    "cid": "user_upload",
    "smiles": ref_smiles,
    "pdbqt_path": f"{PROJ_DIR}/results/ligand_user_upload/ligand_user_upload.pdbqt",
    "sim_morgan": 1.000,
    "sim_maccs": 1.000
}

if len(valid_compounds) == 0 or valid_compounds[0]['id'] != 'ligand_user_upload':
    valid_compounds.insert(0, ligand_user_data)

print(f"[系統] 開始高通量 GPU 對接 (共 {len(valid_compounds)} 個分子)...")
print(f"[系統] 統一使用新質心: X={new_cx:.2f}, Y={new_cy:.2f}, Z={new_cz:.2f}")
print(f"[系統] 繼承長方體尺寸: X={sx}, Y={sy}, Z={sz}")

# ★ 防護 1：檢查受體是否存在，避免開局就崩潰
receptor_source = f"{PROJ_DIR}/receptor.pdbqt"
if not os.path.exists(receptor_source):
    raise FileNotFoundError(f"[錯誤] 找不到受體檔案: {receptor_source}，對接無法進行。")
shutil.copy(receptor_source, "/content/run_receptor.pdbqt")

for idx, comp in enumerate(valid_compounds):
    mol_id = comp['id']
    orig_pdbqt = comp['pdbqt_path']
    run_pdbqt = f"/content/run_{mol_id}.pdbqt"

    label = "Reference Redock" if mol_id == "ligand_user_upload" else "Analog"
    print(f"  -> [{idx+1}/{len(valid_compounds)}] Docking {mol_id} ({label}) ...", end="", flush=True)

    # ★ 防護 2：確保配體的 PDBQT 真的存在，否則優雅跳過
    if not os.path.exists(orig_pdbqt):
        print(" [略過] 找不到對應的 PDBQT 結構檔。")
        comp['affinity'] = 999.0  # 設為極端差的數值，確保排序時墊底
        continue

    try:
        shutil.copy(orig_pdbqt, run_pdbqt)
    except Exception as e:
        print(f" [略過] 複製檔案時發生異常: {e}")
        comp['affinity'] = 999.0
        continue

    cmd = [
        VINA_GPU_PATH,
        "--receptor", "run_receptor.pdbqt",
        "--ligand", run_pdbqt,
        "--center_x", str(new_cx), "--center_y", str(new_cy), "--center_z", str(new_cz),
        "--size_x", str(sx), "--size_y", str(sy), "--size_z", str(sz),
        "--thread", "8000"
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    run_out = f"/content/run_{mol_id}_out.pdbqt"
    final_out = f"{PROJ_DIR}/results/{mol_id}/{mol_id}_out.pdbqt"

    if os.path.exists(run_out):
        shutil.move(run_out, final_out)
        with open(final_out, "r") as f:
            for line in f:
                if line.startswith("REMARK VINA RESULT"):
                    comp['affinity'] = float(line.split()[3])
                    break
        print(f" 完成 (Affinity: {comp.get('affinity', 'N/A')})")
    else:
        # ★ 防護 3：即使 Vina 跑到一半崩潰沒產出檔案，也給予 999.0 確保不會報錯
        print(" 失敗 (Vina 運算未產出結果)")
        comp['affinity'] = 999.0

    if os.path.exists(run_pdbqt):
        os.remove(run_pdbqt)

print("\n[系統] 所有批次對接任務已完成。")

[系統] 開始高通量 GPU 對接 (共 51 個分子)...
[系統] 統一使用新質心: X=12.71, Y=-5.06, Z=22.55
[系統] 繼承長方體尺寸: X=30.0, Y=30.0, Z=30.0
  -> [1/51] Docking ligand_user_upload (Reference Redock) ... 完成 (Affinity: -9.1)
  -> [2/51] Docking ligand_CID_5287836 (Analog) ... 完成 (Affinity: -9.1)
  -> [3/51] Docking ligand_CID_1631 (Analog) ... 完成 (Affinity: -9.1)
  -> [4/51] Docking ligand_CID_10413130 (Analog) ... 完成 (Affinity: -8.9)
  -> [5/51] Docking ligand_CID_133792590 (Analog) ... 完成 (Affinity: -8.0)
  -> [6/51] Docking ligand_CID_87534536 (Analog) ... 完成 (Affinity: -8.5)
  -> [7/51] Docking ligand_CID_44376754 (Analog) ... 完成 (Affinity: -8.8)
  -> [8/51] Docking ligand_CID_44376712 (Analog) ... 完成 (Affinity: -9.0)
  -> [9/51] Docking ligand_CID_101104399 (Analog) ... 完成 (Affinity: -8.4)
  -> [10/51] Docking ligand_CID_44376651 (Analog) ... 完成 (Affinity: -8.5)
  -> [11/51] Docking ligand_CID_445296 (Analog) ... 完成 (Affinity: -8.6)
  -> [12/51] Docking ligand_CID_44376713 (Analog) ... 完成 (Affinity: -8.5)
  -> [13

### 結果彙整、報告生成與乾淨打包
功能說明：解析所有對接結果，依據熱力學親和力 (Affinity) 排序，並自動匯出包含雙重相似度指標的 CSV 分析報表，最終打包下載。

核心設計：

1. 資料清洗與報表優化：自動將底層防呆數值 (999.0) 轉換為直覺的 Failed 標記，提升資料可讀性與後續 Pandas / Excel 處理的便利性。

2. 無塵室打包機制 (Clean-room Archiving)：在壓縮 ZIP 封存檔前，系統會強制清除所有記憶體暫存、原始上傳冗餘與沙盒殘留檔案，確保輸出的壓縮包達到最低容量，且內容僅包含純粹、立即可用的高通量篩選結果。

In [ ]:
import pandas as pd
from IPython.display import display
import time
import os
import shutil
from google.colab import files # 稍微優化 import 寫法

# 1. 確保照 Affinity 排序 (失敗的 999.0 會自動排到最後面)
sorted_compounds = sorted(valid_compounds, key=lambda x: x.get('affinity', 999.0))

results_data = []
for comp in sorted_compounds:
    serial = "user_upload" if comp['id'] == "ligand_user_upload" else f"CID_{comp['cid']}"

    # ★ 微調：把用來墊底的 999.0 轉換為漂亮的 "Failed" 字樣
    raw_affinity = comp.get('affinity', 999.0)
    display_affinity = "Failed" if raw_affinity >= 999.0 else raw_affinity

    results_data.append({
        "ID": serial,
        "RDKit_SMILES": comp['smiles'],
        "Morgan_Similarity(%)": f"{comp['sim_morgan']*100:.2f}",
        "MACCS_Similarity(%)": f"{comp['sim_maccs']*100:.2f}",
        "Affinity(kcal/mol)": display_affinity
    })

df = pd.DataFrame(results_data)
csv_path = f"{PROJ_DIR}/results/HTVS_Summary_Report.csv"
df.to_csv(csv_path, index=False, encoding='utf-8-sig')

print("[系統] 實驗結果摘要 (前 10 名):")
display(df.head(10))

print(f"\n[系統] 正在清理暫存檔與乾淨打包專案結果...")

# 2. 移除 PROJ_DIR 根目錄下的暫存檔與原始上傳檔，確保打包只包含 results 與 receptor.pdbqt
for item in os.listdir(PROJ_DIR):
    item_path = os.path.join(PROJ_DIR, item)
    if item not in ['results', 'receptor.pdbqt']:
        if os.path.isfile(item_path):
            os.remove(item_path)
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)

# 3. 壓縮打包
timestamp = time.strftime("%Y%m%d_%H%M")
zip_filename = f"/content/HTVS_Results_{timestamp}.zip"

# shutil.make_archive 會自動把 PROJ_DIR 裡面的東西打包成 zip
shutil.make_archive(zip_filename.replace('.zip', ''), 'zip', PROJ_DIR)

# 4. 觸發下載
if os.path.exists(zip_filename):
    print(f"[系統] 打包成功 ({os.path.getsize(zip_filename)/1024/1024:.2f} MB)。正在觸發下載...")
    files.download(zip_filename)

[系統] 實驗結果摘要 (前 10 名):


,ID,RDKit_SMILES,Morgan_Similarity(%),MACCS_Similarity(%),Affinity(kcal/mol)
0,CID_5287836,N=C(N)c1cccc(C(=O)N[C@@H](C(=O)N2CCC(C(=O)c3cc...,87.04,100.00,Failed
1,CID_1631,N=C(N)c1cccc(C(=O)NC(C(=O)N2CCC(C(=O)c3ccccc3)...,87.04,100.00,Failed
2,CID_10413130,N=C(N)c1cccc(C(=O)N[C@@H](C(=O)N2CCC(C(=O)c3cc...,78.33,91.84,Failed
3,CID_133792590,O=C(NC(C(=O)N1CCC(C(=O)c2ccccc2)CC1)c1ccccc1)c...,73.08,84.44,Failed
4,CID_87534536,N/N=C\c1cccc(C(=O)NC(C(=O)N2CCC(C(=O)c3ccccc3)...,66.67,85.71,Failed
5,CID_44376754,N=C(N)c1cccc(C(=O)N[C@@H](C(=O)N2CCC(Cc3ccccc3...,66.13,93.48,Failed
6,CID_44376712,N=C(N)c1cccc(C(=O)N[C@@H](C(=O)N2CCN(C(=O)c3cc...,64.41,95.65,Failed
7,CID_101104399,N=C(N)c1cccc(C(=O)N[C@H](Cc2ccccc2)C(=O)N2CCC(...,63.49,95.74,Failed
8,CID_44376651,N=C(N)c1cccc(C(=O)N[C@@H](C(=O)N2CCN(c3ccccc3)...,62.30,87.76,Failed
9,CID_445296,N=C(N)c1cccc(C(=O)N[C@@H](C(=O)N2C=C[C@@H](C(=...,61.54,87.23,Failed



[系統] 正在清理暫存檔與乾淨打包專案結果...
[系統] 打包成功 (0.26 MB)。正在觸發下載...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>